# Gold — `dim_grupos`

Criação da dimensão `dim_grupos` diretamente na camada **Gold**.

Como este é um dicionário pequeno, controlado e mantido no próprio projeto, não é necessário gerar CSV e fazer uma etapa intermediária de Volume. O notebook cria o DataFrame e persiste diretamente como tabela Delta.

Tabela final: `pnd2025.gold.dim_grupos`

In [0]:
# Célula 1 — Definir parâmetros

CATALOG = "pnd2025"
SCHEMA_GOLD = "gold"
TABELA_GOLD = f"{CATALOG}.{SCHEMA_GOLD}.dim_grupos"

print(f"Tabela destino: {TABELA_GOLD}")

In [0]:
# Célula 2 — Criar o schema Gold, caso ainda não exista

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")
print(f"Schema disponível: {CATALOG}.{SCHEMA_GOLD}")

In [0]:
# Célula 3 — Definir os grupos

dados_grupos = [(702, 'MATEMÁTICA (LICENCIATURA)'), (904, 'LETRAS - PORTUGUÊS (LICENCIATURA)'), (905, 'LETRAS - PORTUGUÊS E INGLÊS (LICENCIATURA)'), (906, 'LETRAS - PORTUGUÊS E ESPANHOL (LICENCIATURA)'), (1402, 'FÍSICA (LICENCIATURA)'), (1502, 'QUÍMICA (LICENCIATURA)'), (1602, 'CIÊNCIAS BIOLÓGICAS (LICENCIATURA)'), (2001, 'PEDAGOGIA (LICENCIATURA)'), (2402, 'HISTÓRIA (LICENCIATURA)'), (2501, 'ARTES VISUAIS (LICENCIATURA)'), (3002, 'GEOGRAFIA (LICENCIATURA)'), (3202, 'FILOSOFIA (LICENCIATURA)'), (3502, 'EDUCAÇÃO FÍSICA (LICENCIATURA)'), (4005, 'CIÊNCIA DA COMPUTAÇÃO (LICENCIATURA)'), (4301, 'MÚSICA (LICENCIATURA)'), (5402, 'CIÊNCIAS SOCIAIS (LICENCIATURA)'), (6407, 'LETRAS - INGLÊS (LICENCIATURA)')]


In [0]:
# Célula 4 — Criar o DataFrame da dimensão

df_dim_grupos = spark.createDataFrame(
    dados_grupos,
    ["co_grupo", "ds_grupo"]
)

display(df_dim_grupos)

In [0]:
# Célula 5 — Validar a dimensão antes da gravação

duplicados = (
    df_dim_grupos
    .groupBy("co_grupo")
    .count()
    .filter("count > 1")
)

assert duplicados.count() == 0, "Existem códigos co_grupo duplicados."
assert df_dim_grupos.filter("co_grupo IS NULL").count() == 0, "co_grupo possui nulos."
assert df_dim_grupos.filter("ds_grupo IS NULL").count() == 0, "ds_grupo possui nulos."

print(f"Quantidade de grupos: {df_dim_grupos.count()}")
print("Validações aprovadas.")

## 6. Gravar diretamente na Gold

Nesta etapa eliminamos o CSV intermediário.

O Volume seria necessário se os dados fossem recebidos como arquivo externo ou precisassem ser disponibilizados como landing. Para uma dimensão pequena e estática como esta, a tabela Delta Gold é mais simples.

In [0]:
# Célula 6 — Persistir como tabela Delta na Gold

(
    df_dim_grupos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD)
)

print(f"Tabela criada/atualizada: {TABELA_GOLD}")

In [0]:
# Célula 7 — Validar a tabela Gold

df_gold = spark.table(TABELA_GOLD)

print(f"Registros: {df_gold.count()}")
print(f"Colunas: {df_gold.columns}")
df_gold.printSchema()

In [0]:
# Célula 8 — Visualizar resultado final

display(df_gold.orderBy("co_grupo"))

In [0]:
# Célula 9 — Consultar histórico Delta

spark.sql(
    f"DESCRIBE HISTORY {TABELA_GOLD}"
).show(truncate=False)

In [0]:
# Etapa final — Padronizar e gravar ds_grupo na Gold

from pyspark.sql.functions import initcap, col

df_dim_grupos = df_dim_grupos.withColumn(
    "ds_grupo",
    initcap(col("ds_grupo"))
)

# Grava a versão final na tabela Gold
(
    df_dim_grupos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("pnd2025.gold.dim_grupos")
)

# Conferência final
display(
    spark.table("pnd2025.gold.dim_grupos")
    .orderBy("co_grupo")
)

## Resultado final

```text
pnd2025
└── gold
    └── dim_grupos
```

A dimensão está pronta para ser utilizada em joins com as tabelas Silver durante a construção da Gold.